### Summary:
In this notebook we will process the SCARlink, ABC and Cicero links into a common format (bedpe files with gene TSS from gencode v32 coordinates). These functions are designed to work in an automated fashion and be easy to run on many outputs from different cell types. In this notebook we will also process both the significant and non-significant links files from all methods for downstream analyses when building background sets for enrichment. 

### Standardized filtering and formatting for all methods:
- Link filtering: genes TPM > 1 in cell type, cRE within 250kb of gene TSS, cREs are union peaks, only CP (cRE-promoter) links (this applies to both sig and bg link sets from each method)
- Output file format: bedpe with cRE coords, gene coords, gene name, score
- Background link sets do not have significant links

# 0. Basic Preparation

In [1]:
# Import necessary libraries
suppressMessages(library(tidyverse))
suppressMessages(library(dplyr))
suppressMessages(library(stringr))
suppressMessages(library(tictoc))
suppressMessages(library(parallel))

In [2]:
# Define celltypes list
celltypes <- c('beta','alpha','delta','gamma','ductal','acinar')

### Define necessary reference files

In [3]:
# Read in the gene coords reference file
ref_df <- read.table('/nfs/lab/ABC/references/gene_coords.gencodev32.hg38.bed', sep='\t', header=FALSE) #read in gene coords ref
colnames(ref_df) <- c('chr','start','end','gene','misc','strand')

# Read in the TSS500bp version of the gene coords reference file
TSS_ref_fp <- '/nfs/lab/ABC/references/gene_coords.gencodev32.hg38.TSS0index.bed'
TSS_ref_df <- read.table(TSS_ref_fp, sep='\t')
colnames(TSS_ref_df) <- c('chr','start','end','gene','strand')

In [4]:
# All CREs (merged list)
cres_fp <- '/nfs/lab/projects/multiomic_islet/outputs/multiome/call_peaks/240304_union_peaks/union_peaks.sort_3col.bed'

# Celltype specific CREs file path = ct_cres_prefix + celltype + ct_cres_suffix
ct_cres_prefix <- '/nfs/lab/projects/multiomic_islet/outputs/multiome/call_peaks/240304_union_peaks/celltype_union_peaks/'
ct_cres_suffix <- '.union_peaks.cut.bed'

In [5]:
# Expressed genes file path = tpm_prefix + celltype + tpm_suffix
tpm_prefix <- "/nfs/lab/projects/multiomic_islet/outputs/multiome/rna_profiles/recluster_final_clustering_v2/major_celltype_TPM/"
tpm_suffix <- "_expressed_genes_TPM1.txt"

### Establish file naming practices for links files

In [6]:
gen_outdir <- "/nfs/lab/projects/multiomic_islet/outputs/revisions_outputs/cRE-gene_links/links"

In [17]:
# SMORES method links files = sm_prefix + celltype + '/' + celltype + sm_suffix
sm_prefix <- '/nfs/lab/hmummey/multiomic_islet/intermediates/230228_SMORES_PP_investigation/'
sm_suffix1 <- '_sig_CP_links.bedpe.gz'
sm_suffix2 <- '_all_CP_links.bedpe.gz'

# Outdir for CP-only SMORES links
sm_outdir <- file.path(gen_outdir,'SMORES')

In [8]:
# SCARlink raw outputs
scarlink_dir <- '/nfs/lab/tscc/hmummey/multiomic_islet/241122_15k_SCARlink_rerun/outputs'
scarlink_fp <- file.path(scarlink_dir, 'gene_linked_tiles_celltype.csv.gz')

# Outdir for cell type SCARlink links
scarlink_outdir <- file.path(gen_outdir,'SCARlink')

In [9]:
# Celltype specific ABC files = abc_prefix + celltype + abc_suffix
abc_prefix <- '/nfs/lab/projects/multiomic_islet/outputs/multiome/cRE-gene_links/run_abc/230110_allCTs/outputs/230116_H3K27ac_CTs/'
abc_suffix <- '/Prediction/EnhancerPredictions.hg38.mapped.bedpe'

# Outdir for reformatted ABC links
abc_outdir <- file.path(gen_outdir,'ABC')

In [10]:
# Celltype specific Cicero files = cic_prefix + celltype + cic_suffix
cic_prefix <- '/nfs/lab/projects/multiomic_islet/outputs/multiome/cRE-gene_links/cicero/241118_union_peaks/Cicero_links.'
cic_suffix <- '.above0.05.dedup.bedpe'

# Outdir for reformatted ABC links
cic_outdir <- file.path(gen_outdir,'Cicero')

In [11]:
# Directory to write the 3 method combined background file to
overlap_outdir <- file.path(gen_outdir,'combined')

## General use functions

In [13]:
overlap_with_peaks <- function(celltype, links_df, links_outdir){
    # Get the left side coordinates from each link (unique and sorted alphanumerically)
    sites <- links_df[,c(1,2,3)]
    colnames(sites) <- c('V1','V2','V3')
    chr_names <- c(paste("chr",seq(1:22),sep=''),'chrX','chrY')
    sites_cut <- sites[sites$V1 %in% chr_names,]
    all_sites_list = paste(sites_cut$V1,sites_cut$V2,sites_cut$V3,sep='_')
    all_sites_list <- str_sort(unique(all_sites_list), numeric=TRUE)
    
    # Output a bed file of these coordinates 
    fin_df <- as.data.frame(str_split_fixed(all_sites_list, '_', n=3))
    out_fp <- file.path(links_outdir, sprintf('%s_all_linked_sites.bed', celltype))
    write.table(fin_df, out_fp, sep='\t', row.names=FALSE, col.names=FALSE, quote=FALSE)
    
    # Overlap with celltype CREs list using bedtools in the terminal
    # Previously used -f but this loses some overlaps (if peak is larger than mapping peak)
    cres_fp <- paste0(ct_cres_prefix, celltype, ct_cres_suffix)
    overlap_fp <- file.path(links_outdir, sprintf('%s_all_linked_sites.union_peaks.overlap.bed', celltype))
    cmd <- paste('bedtools intersect -a', out_fp, '-b', cres_fp, '-wo >', overlap_fp, sep=' ')
    system(cmd)   
    return(overlap_fp)
}

In [14]:
### Function to extract a gene's TSS from the reference file
get_TSS <- function(gene){
    if (gene %in% ref_df$gene == TRUE){
        ref_df_cut = ref_df[ref_df$gene ==gene,]
        if (ref_df_cut$strand == '-'){
            tss = max(c(ref_df_cut$start,ref_df_cut$end))
        } else {
            tss = min((c(ref_df_cut$start,ref_df_cut$end)))
        }
        return(tss)
    } else {
        return(NA)
    }
}

In [15]:
### Function to calculate link distances from a bedpe style dataframe row
calc_link_distance <- function(link_df_row){
    CRE_start <- as.integer(link_df_row[2])
    CRE_end <- as.integer(link_df_row[3])
    gene_start <- as.integer(link_df_row[5])
    CRE_center <- CRE_start + (CRE_end - CRE_start)/2
    distance <- abs(CRE_center - gene_start)
    return(distance)
}

# 1. Reformat SMORES Links

## 1a. Reformat and filter significant links

### Functions

In [25]:
### Function to map SMORES links to union cREs, and then filter to consistent format
filt_smores <- function(celltype, sm_fp, outdir, max_dist=250000){
    #first read in the links from the CP cut files and map to union peaks
    links <- read.table(sm_fp, sep='\t')
    overlap_fp <- overlap_with_peaks(celltype, links, outdir)

    #now read in overlap results and finish mapping
    sm_overlap <- read.table(overlap_fp, sep='\t')
    sm_overlap$peak1 <- paste(sm_overlap$V1, sm_overlap$V2, sm_overlap$V3, sep='-')
    sm_overlap$peak2 <- paste(sm_overlap$V4, sm_overlap$V5, sm_overlap$V6, sep='-')
    
    # Decide between peaks that map to multiple peaks (based on bp overlap)
    # Sort by the overlap # and then remove duplicate peak1
    sm_overlap_sort <- sm_overlap[order(sm_overlap$V7, decreasing = TRUE),]
    sm_overlap_cut <- sm_overlap_sort[!duplicated(sm_overlap_sort$peak1),c(8,9)]
    
    # Read in ABC output and map CRE coords
    links$peak1 <- paste(links$V1, links$V2, links$V3, sep="-")
    links_overlap <- merge(links, sm_overlap_cut, by='peak1')
    links_nomap <- links[!links$peak1 %in% links_overlap$peak1,seq(1,8)]

    # Print out statistics
    print(celltype)
    print(paste('Number of unique peaks before mapping: ',length(unique(links$peak1)), sep=''))
    print(paste('Number of mapped links: ', dim(links_overlap)[1], '/', dim(links)[1], sep=''))
    print(paste('Number of unmapped links: ', dim(links_nomap)[1], '/', dim(links)[1], sep=''))

    # Now make final df where the CRE coords cols are altered to be the mapped ones
    # Excludes any links for which the CRE did not map
    fin_sm_df <- cbind(str_split_fixed(links_overlap$peak2, '-', 3), links_overlap[,c(5,6,7,8,9)])
    
    # Apply final filters: gene TPM > 1 in cell type; distance <= 250kb from TSS
    tpm_fp <- paste0(tpm_prefix,celltype,tpm_suffix)
    tpm1_genes <- scan(tpm_fp, what='', sep='\t')
    fin_sm_df$distance <- apply(fin_sm_df, 1, calc_link_distance)
    fin_sm_df <- fin_sm_df %>% mutate(expressed=ifelse(V7 %in% tpm1_genes,'y','n')) %>%
                        subset(expressed == 'y' & distance < max_dist) %>%
                        select(-c(distance,expressed))
    print(paste('Number of mapped links passing filters: ', dim(fin_sm_df)[1], '/', dim(links_overlap)[1], sep=''))
    print('')
    
    #write to file
    mapped_sm_fp <- file.path(outdir, sprintf('%s_mapped_links.bedpe',celltype))
    write.table(fin_sm_df, mapped_sm_fp, sep='\t', row.names=FALSE, col.names=FALSE, quote=FALSE)
}

### Run functions to classify links

In [27]:
# Map links to union peaks and filter
for(celltype in celltypes){
    links_fp <- paste0(sm_prefix,celltype,sm_suffix1)
    filt_smores(celltype, links_fp, sm_outdir)
}

[1] "beta"
[1] "Number of unique peaks before mapping: 57884"
[1] "Number of mapped links: 114381/114389"
[1] "Number of unmapped links: 8/114389"
[1] "Number of mapped links passing filters: 43487/114381"
[1] ""
[1] "alpha"
[1] "Number of unique peaks before mapping: 44300"
[1] "Number of mapped links: 71004/71011"
[1] "Number of unmapped links: 7/71011"
[1] "Number of mapped links passing filters: 27881/71004"
[1] ""
[1] "delta"
[1] "Number of unique peaks before mapping: 25821"
[1] "Number of mapped links: 32135/32138"
[1] "Number of unmapped links: 3/32138"
[1] "Number of mapped links passing filters: 11616/32135"
[1] ""
[1] "gamma"
[1] "Number of unique peaks before mapping: 23743"
[1] "Number of mapped links: 29401/29405"
[1] "Number of unmapped links: 4/29405"
[1] "Number of mapped links passing filters: 9647/29401"
[1] ""
[1] "ductal"
[1] "Number of unique peaks before mapping: 40881"
[1] "Number of mapped links: 58411/58417"
[1] "Number of unmapped links: 6/58417"
[1] "Number 

## 1b. Reformat background links

In [28]:
# Map to union peaks and filter (also make sure none of the sig links are in these files)
# Map links to union peaks and filter
for(celltype in celltypes){
    links_fp <- paste0(sm_prefix,celltype,sm_suffix2)
    filt_smores(celltype, links_fp, file.path(sm_outdir,'nonsig_links'))
}

[1] "beta"
[1] "Number of unique peaks before mapping: 94375"
[1] "Number of mapped links: 1808175/1808345"
[1] "Number of unmapped links: 170/1808345"
[1] "Number of mapped links passing filters: 496362/1808175"
[1] ""
[1] "alpha"
[1] "Number of unique peaks before mapping: 96287"
[1] "Number of mapped links: 1985909/1986088"
[1] "Number of unmapped links: 179/1986088"
[1] "Number of mapped links passing filters: 548602/1985909"
[1] ""
[1] "delta"
[1] "Number of unique peaks before mapping: 95119"
[1] "Number of mapped links: 1933886/1934106"
[1] "Number of unmapped links: 220/1934106"
[1] "Number of mapped links passing filters: 530220/1933886"
[1] ""
[1] "gamma"
[1] "Number of unique peaks before mapping: 91381"
[1] "Number of mapped links: 1884967/1885225"
[1] "Number of unmapped links: 258/1885225"
[1] "Number of mapped links passing filters: 523162/1884967"
[1] ""
[1] "ductal"
[1] "Number of unique peaks before mapping: 100402"
[1] "Number of mapped links: 2369461/2369623"
[1] "N

# 2. Reformat SCARlink Links

## 2a. Reformat all links

### Functions

In [30]:
### Extract and reformat SCARlink sig links per cell type
### sig_threshold is for FDR significant, default is FDR<0.1
reformat_scarlink <- function(scarlink, ct, sig_threshold=0.1, overlap_threshold=200, max_dist=250000){
    #subset down to cell type of interest and add in gene TSS coords (TSS, TSS+1)
    links <- scarlink %>% filter(celltype == ct) %>%
                left_join(TSS_ref_df, by='gene') %>%
                relocate(c('chr.y','start.y','end.y','gene','regression_coef'), .after=end.x)

    #overlap tiles from scarlink outputs with peaks
    overlap_fp <- overlap_with_peaks(ct, links, scarlink_outdir)

    #now read in the overlap results and map links coords to union peaks
    link_overlap <- read.table(overlap_fp, sep='\t')
    link_overlap <- link_overlap %>% mutate(peak1=paste(V1,V2,V3,sep='-'), peak2=paste(V4,V5,V6, sep='-'))

    # Decide between peaks that map to multiple peaks (based on bp overlap -- but if close leave both)
    # Sort by the overlap # and then remove duplicate peak1 if V7 < 200
    link_overlap <- link_overlap %>% arrange(-V7)
    link_overlap_cut <- link_overlap[!duplicated(link_overlap$peak1) | link_overlap$V7 > overlap_threshold,c(8,9)]

    #now use this to map links
    links$peak1 <- paste(links$chr.x, links$start.x, links$end.x, sep='-')
    links_df_overlap <- merge(links, link_overlap_cut, by='peak1')
    nomap <- links[!links$peak1 %in% links_df_overlap$peak1,seq(1,8)]

    # Print out statistics
    print(paste('Number of unique peaks before mapping: ',length(unique(links$peak1)), sep=''))
    print(paste('Number of mapped unique links: ', dim(links_df_overlap)[1], '/', dim(links)[1], sep=''))
    print(paste('Number of unmapped links: ', dim(nomap)[1], '/', dim(links)[1], sep=''))
    print('')  

    #for instances where one peaks overlaps multiple tiles predicted to regulate the same gene, just take the max regression coef
    links_df_overlap <- links_df_overlap %>% arrange(peak2, gene, -regression_coef) %>%
                        distinct(peak2, gene, .keep_all=TRUE) %>%
                        relocate(peak1, .before=peak2)

    #add in final filtering (gene TPM > 1 in cell type; distance <= 250kb from TSS)
    tpm_fp <- paste0(tpm_prefix,ct,tpm_suffix)
    tpm1_genes <- scan(tpm_fp, what='', sep='\t')
    links_df_overlap$distance <- apply(links_df_overlap, 1, calc_link_distance)
    dim(links_df_overlap)
    links_df_overlap <- links_df_overlap %>% mutate(expressed=ifelse(gene %in% tpm1_genes,'y','n')) %>%
                        subset(expressed == 'y' & distance < max_dist)
    
    # Now make final output df where the CRE coords cols are altered to be the mapped ones
    fin_links <- cbind(str_split_fixed(links_df_overlap$peak2, '-', 3), links_df_overlap[,c(4,5,6,7,8,12,13,15,14)])
    all_scarlinks_fp <- file.path(scarlink_outdir, sprintf('%s_mapped_ALL_links.bedpe',ct))
    write.table(fin_links, all_scarlinks_fp, sep='\t', row.names=FALSE, col.names=FALSE, quote=FALSE)
}

### Run Functions on all cell types

In [15]:
#read in the combined scarlink outputs file once bc super big
scarlink <- read.table(scarlink_fp, sep='\t', header=1)

In [31]:
for(celltype in celltypes){
    print(paste(celltype, Sys.time()))
    reformat_scarlink(scarlink, celltype)
}

[1] "beta 2024-12-02 15:56:02"
[1] "Number of unique peaks before mapping: 4008191"
[1] "Number of mapped unique links: 495388/9634251"
[1] "Number of unmapped links: 9140371/9634251"
[1] ""
[1] "alpha 2024-12-02 15:57:46"
[1] "Number of unique peaks before mapping: 4008191"
[1] "Number of mapped unique links: 511926/9634251"
[1] "Number of unmapped links: 9123910/9634251"
[1] ""
[1] "delta 2024-12-02 15:59:25"
[1] "Number of unique peaks before mapping: 4008191"
[1] "Number of mapped unique links: 495941/9634251"
[1] "Number of unmapped links: 9139862/9634251"
[1] ""
[1] "gamma 2024-12-02 16:01:05"
[1] "Number of unique peaks before mapping: 4008191"
[1] "Number of mapped unique links: 495380/9634251"
[1] "Number of unmapped links: 9140474/9634251"
[1] ""
[1] "ductal 2024-12-02 16:02:41"
[1] "Number of unique peaks before mapping: 4008191"
[1] "Number of mapped unique links: 514354/9634251"
[1] "Number of unmapped links: 9121385/9634251"
[1] ""
[1] "acinar 2024-12-02 16:04:21"
[1] "Nu

## 2b. Separate links into sig and background sets

### Functions

In [36]:
### Extract and reformat SCARlink sig links per cell type
### sig_threshold is for FDR significant, default is FDR<0.1
get_sig_scarlink <- function(ct, fdr_threshold=0.1, z_threshold=0.5, corr_threshold=0.1){
    #read in the mapped all links
    all_scarlinks_fp <- file.path(scarlink_outdir, sprintf('%s_mapped_ALL_links.bedpe',ct))
    links <- read.table(all_scarlinks_fp, sep='\t')
    colnames(links) <- c('chr1','start1','end1','chr2','start2','end2','gene',
                         'regression_coef','zscore','pvalue','FDR','correlation')
    
    #subset down to significant links remove the p/qvalue cols and write to a file
    sig_links <- links %>% filter(FDR < fdr_threshold & zscore > z_threshold & correlation > corr_threshold) %>%
                    select(-c(pvalue,FDR))
    out_fp <- file.path(scarlink_outdir, sprintf('%s_mapped_SIG_links.bedpe',ct))
    write.table(sig_links, out_fp, sep='\t', row.names=F, col.names=F, quote=F)

    #also write bg file for all links not sig
    bg_links <- links %>% filter(FDR >= fdr_threshold | zscore <= z_threshold | correlation <= corr_threshold) %>%
                    select(-c(pvalue,FDR))
    out_fp2 <- file.path(scarlink_outdir, sprintf('%s_mapped_BG_links.bedpe',ct))
    write.table(bg_links, out_fp2, sep='\t', row.names=F, col.names=F, quote=F)

    #print out a little summary of # of links
    print(paste('Number of all mapped peak links: ',dim(links)[1], sep=''))
    print(paste('Number of significant mapped peak links: ', dim(sig_links)[1], '/', dim(links)[1], sep=''))
    print('')   
}

### Run Functions on all cell types

In [37]:
for(celltype in celltypes){
    print(paste(celltype, Sys.time()))
    get_sig_scarlink(celltype)
}

[1] "beta 2024-12-03 11:29:56"
[1] "Number of all mapped peak links: 273336"
[1] "Number of significant mapped peak links: 11218/273336"
[1] ""
[1] "alpha 2024-12-03 11:29:59"
[1] "Number of all mapped peak links: 284175"
[1] "Number of significant mapped peak links: 9774/284175"
[1] ""
[1] "delta 2024-12-03 11:30:02"
[1] "Number of all mapped peak links: 274847"
[1] "Number of significant mapped peak links: 2918/274847"
[1] ""
[1] "gamma 2024-12-03 11:30:05"
[1] "Number of all mapped peak links: 275759"
[1] "Number of significant mapped peak links: 1492/275759"
[1] ""
[1] "ductal 2024-12-03 11:30:08"
[1] "Number of all mapped peak links: 285666"
[1] "Number of significant mapped peak links: 2963/285666"
[1] ""
[1] "acinar 2024-12-03 11:30:11"
[1] "Number of all mapped peak links: 267471"
[1] "Number of significant mapped peak links: 5799/267471"
[1] ""


# 3. Reformat ABC Links
Because ABC maps peaks to hg19, runs predictions, and then maps back the cREs the cRE coords may no longer match our union peaks, so we need to map to them.
Considerations:
- The bedtools intersect command to overlap ABC CREs with the peak calls list sometimes returns double outputs (aka one CRE is mapped to the same peak twice)

## 3a. Reformat significant links

### Functions

In [19]:
### Function to map ABC cRE coords to overlapping peaks and output reformatted dataframe
map_to_overlap_peaks <- function(celltype, abc_fp, abc_outdir, max_dist=250000){
    # Read in the bedtools intersect output, cut out repeats, and get map
    overlap_fp <- file.path(abc_outdir, sprintf('%s_all_linked_sites.union_peaks.overlap.bed', celltype))
    abc_overlap <- read.table(overlap_fp, sep='\t')
    abc_overlap$peak1 <- paste(abc_overlap$V1, abc_overlap$V2, abc_overlap$V3, sep='-')
    abc_overlap$peak2 <- paste(abc_overlap$V4, abc_overlap$V5, abc_overlap$V6, sep='-')
    
    # Decide between peaks that map to multiple peaks (based on bp overlap)
    # Sort by the overlap # and then remove duplicate peak1
    abc_overlap_sort <- abc_overlap[order(abc_overlap$V7, decreasing = TRUE),]
    abc_overlap_cut <- abc_overlap_sort[!duplicated(abc_overlap_sort$peak1),c(8,9)]
    
    # Read in ABC output and map CRE coords
    abc_df <- read.table(abc_fp, sep='\t')
    abc_df$peak1 <- paste(abc_df$V1, abc_df$V2, abc_df$V3, sep="-")
    abc_df_overlap <- merge(abc_df, abc_overlap_cut, by='peak1')
    abc_nomap <- abc_df[!abc_df$peak1 %in% abc_overlap$peak1,seq(1,8)]

    # Print out statistics
    print(celltype)
    print(paste('Number of unique peaks before mapping: ',length(unique(abc_df$peak1)), sep=''))
    print(paste('Number of mapped links: ', dim(abc_df_overlap)[1], '/', dim(abc_df)[1], sep=''))
    print(paste('Number of unmapped links: ', dim(abc_nomap)[1], '/', dim(abc_df)[1], sep=''))

    # Now make final ABC df where the CRE coords cols are altered to be the mapped ones
    # Excludes any links for which the CRE did not map
    fin_abc_df <- cbind(str_split_fixed(abc_df_overlap$peak2, '-', 3), abc_df_overlap[,c(5,6,7,8,9)])

    # Apply final filters: gene TPM > 1 in cell type; distance <= 250kb from TSS
    tpm_fp <- paste0(tpm_prefix,celltype,tpm_suffix)
    tpm1_genes <- scan(tpm_fp, what='', sep='\t')
    fin_abc_df$distance <- apply(fin_abc_df, 1, calc_link_distance)
    fin_abc_df <- fin_abc_df %>% mutate(expressed=ifelse(V7 %in% tpm1_genes,'y','n')) %>%
                        subset(expressed == 'y' & distance < max_dist) %>%
                        select(-c(distance,expressed))
    print(paste('Number of mapped links passing filters: ', dim(fin_abc_df)[1], '/', dim(abc_df)[1], sep=''))
    print('')
    
    #write to file
    mapped_abc_fp <- file.path(abc_outdir, sprintf('%s_mapped_links.bedpe',celltype))
    write.table(fin_abc_df, mapped_abc_fp, sep='\t', row.names=FALSE, col.names=FALSE, quote=FALSE)
}

### Run functions on all celltypes to reformat links

In [20]:
tic()
for (celltype in celltypes){
    abc_fp <- paste0(abc_prefix, celltype, abc_suffix)
    abc_links <- read.table(abc_fp, sep='\t')
    overlap_with_peaks(celltype, abc_links, abc_outdir)
    map_to_overlap_peaks(celltype, abc_fp, abc_outdir)
}
toc()

[1] "beta"
[1] "Number of unique peaks before mapping: 13213"
[1] "Number of mapped links: 39258/39495"
[1] "Number of unmapped links: 237/39495"
[1] "Number of mapped links passing filters: 29333/39495"
[1] ""
[1] "alpha"
[1] "Number of unique peaks before mapping: 14026"
[1] "Number of mapped links: 40821/41043"
[1] "Number of unmapped links: 222/41043"
[1] "Number of mapped links passing filters: 30967/41043"
[1] ""
[1] "delta"
[1] "Number of unique peaks before mapping: 13439"
[1] "Number of mapped links: 40526/40774"
[1] "Number of unmapped links: 248/40774"
[1] "Number of mapped links passing filters: 30269/40774"
[1] ""
[1] "gamma"
[1] "Number of unique peaks before mapping: 13606"
[1] "Number of mapped links: 40562/40868"
[1] "Number of unmapped links: 306/40868"
[1] "Number of mapped links passing filters: 30333/40868"
[1] ""
[1] "ductal"
[1] "Number of unique peaks before mapping: 15859"
[1] "Number of mapped links: 56267/56518"
[1] "Number of unmapped links: 251/56518"
[1] "

## 3b. Reformat non-significant links
Steps:
1. Remove significant links from all links file -- tbh not necessary, but anything that reduces the df size here is valuable
2. Convert to bedpe 
    1. Map ABC CRE coords to peaks set (get garbled from repeated liftover)
    2. Get gene coords from ref file
    3. Reformat as a bedpe
3. Remove links above distance threshold (1Mb)

### First map all hg19 ABC peaks back to hg38 so I can convert the background set to hg38 before mapping to union peaks
Run CrossMap in the terminal:
```
chain="/nfs/lab/ABC/references/hg19ToHg38.over.chain"

for celltype in beta alpha delta gamma acinar ductal; do
    hg19_peaks="/nfs/lab/projects/multiomic_islet/outputs/multiome/cRE-gene_links/run_abc/230110_allCTs/inputs/peak_calls_hg19/"${celltype}"_peaks.narrowPeak"
    hg38_peaks="/nfs/lab/projects/multiomic_islet/outputs/multiome/cRE-gene_links/run_abc/230110_allCTs/inputs/peak_calls_hg19/"${celltype}"_peaks.hg38.narrowPeak"
    CrossMap.py region $chain $hg19_peaks $hg38_peaks
done
```

In [21]:
### Only need to do this once

# #now read in the pre and post mapped peaks and make a mapping file (hg19 coords on the left, hg38 coords on the right)
# peak_dir <- "/nfs/lab/projects/multiomic_islet/outputs/multiome/cRE-gene_links/run_abc/230110_allCTs/inputs/peak_calls_hg19"

# for (celltype in celltypes){
#     #read in both hg19 and hg38 peaks
#     fp1 <- file.path(peak_dir,sprintf('%s_peaks.narrowPeak',celltype))
#     fp2 <- file.path(peak_dir,sprintf('%s_peaks.hg38.narrowPeak',celltype))
#     columns <- c('chr','start','end','peak_name','metric','strand','metric1','metric2','metric3','metric4')
#     df1 <- read.table(fp1, sep='\t', col.names=columns)
#     df2 <- read.table(fp2, sep='\t', col.names=c(columns,'mapping'))

#     #combine hg38 with hg19 (left join with hg19) and write relevant columns to a mapping file
#     fin_df <- df1 %>% left_join(df2, by='peak_name') %>% 
#             select(chr.x,start.x,end.x,chr.y,start.y,end.y) %>%
#             rename(chr=chr.x, start=start.x, end=end.x, chr_hg38=chr.y, start_hg38=start.y, end_hg38=end.y)
#     out_fp <- file.path(peak_dir,sprintf('%s_peaks_map.bedpe',celltype))
#     write.table(fin_df, out_fp, sep='\t', row.names=F, quote=F)
# }

### Functions

In [23]:
### Function to prepare ABC background links reference using R, not bash
### Subset for links with score < 0.02, remove NaN links, select for 1Mb distance max
### Output everything that passes this to a file in bedpe format for comparisons
prep_ABC_background <- function(celltype, abc_dir, outdir, score_threshold=0.02, dist_threshold=250000){
    # Read in all putative links
    all_links_fp <- file.path(abc_dir, 'EnhancerPredictionsAllPutative.txt.gz')
    all_links <- read.table(all_links_fp, sep='\t', header=TRUE)
    print(paste0('Number of links in AllPutative: ', dim(all_links)[1]))

    # 11/18/24: Map the cRE coordinates to hg38 using the map files I made -- 
    # make sure all_links is identical at the end so the rest of this should run fine
    peak_dir <- "/nfs/lab/projects/multiomic_islet/outputs/multiome/cRE-gene_links/run_abc/230110_allCTs/inputs/peak_calls_hg19"
    map_fp <- file.path(peak_dir,sprintf('%s_peaks_map.bedpe',celltype))
    map <- read.table(map_fp, sep='\t', header=TRUE)
    all_links <- all_links %>% left_join(map, by=c('chr','start','end')) %>%
                    subset(!is.na(chr_hg38)) %>%
                    select(-c(chr,start,end)) %>%
                    rename(chr=chr_hg38, start=start_hg38, end=end_hg38) %>%
                    relocate(chr,start,end)
        
    # Remove links passing the significance threshold (0.02) and which are not NaN
    all_links_cut <- all_links[!is.na(all_links$ABC.Score) & all_links$ABC.Score <= score_threshold,]
    print(paste0('Number of links below threshold and not NaN: ', dim(all_links_cut)[1]))
    
    # Merge TSS coords into all_links_cut and select cols to make a bedpe
    merged_df <- all_links_cut %>% rename(gene=TargetGene) %>%
                    left_join(TSS_ref_df, by='gene')
    all_links_bedpe <- merged_df %>% select(chr.x,start.x,end.x,chr.y,start.y,end.y,gene,ABC.Score)

    # Remove links more than 1Mb apart (or whatever distance threshold you choose)
    distances <- unlist(apply(all_links_bedpe, 1, calc_link_distance))
    fin_links <- all_links_bedpe[distances <= dist_threshold,]
    print(paste0('Number of links below distance threshold (', dist_threshold, 'bps): ', dim(fin_links)[1]))
    
    filt_fp <- file.path(abc_dir, 'EnhancerPredictionsAllPutative.filtered.txt')
    write.table(fin_links, filt_fp, sep='\t', row.names=FALSE, col.names=FALSE, quote=FALSE)
    
    # Map CRE coords to peak calls (gzip files)
    overlap_with_peaks(celltype, fin_links, outdir)
    map_to_overlap_peaks(celltype, filt_fp, outdir)
    system(sprintf('gzip %s', filt_fp))
}

### Use functions to prepare the non-significant links files

In [24]:
# ABC prepare files for compare connections -- previously this only took like 5-15 mins per cell type
tic()
for (celltype in celltypes){
    print(paste(celltype, Sys.time()))
    abc_dir <- paste0(abc_prefix, celltype, "/Prediction")
    outdir <- file.path(abc_outdir, 'nonsig_links')
    prep_ABC_background(celltype, abc_dir, outdir)
    print('')
}
toc()

[1] "beta 2024-12-02 12:18:15"
[1] "Number of links in AllPutative: 9334969"
[1] "Number of links below threshold and not NaN: 9166176"
[1] "Number of links below distance threshold (250000bps): 555671"
[1] "beta"
[1] "Number of unique peaks before mapping: 97457"
[1] "Number of mapped links: 499877/555671"
[1] "Number of unmapped links: 55794/555671"
[1] "Number of mapped links passing filters: 499716/555671"
[1] ""
[1] ""
[1] "alpha 2024-12-02 12:22:19"
[1] "Number of links in AllPutative: 10141622"
[1] "Number of links below threshold and not NaN: 9967888"
[1] "Number of links below distance threshold (250000bps): 616546"
[1] "alpha"
[1] "Number of unique peaks before mapping: 101257"
[1] "Number of mapped links: 550679/616546"
[1] "Number of unmapped links: 65867/616546"
[1] "Number of mapped links passing filters: 550491/616546"
[1] ""
[1] ""
[1] "delta 2024-12-02 12:26:42"
[1] "Number of links in AllPutative: 10155074"
[1] "Number of links below threshold and not NaN: 9973861"
[1

# 4. Reformat Cicero links
Considerations:
- Multimapping: some CREs overlap multiple gene promoters, this will count as separate links to all the genes
- Need to make sure CRE coords are in the first 3 columns, may need to swap coords for some rows, if prom2 has the  gene/s
- What to list as the gene coords? The gene coords from ref df or the coords of the CRE that overlaps the genes promoter? --> when we compare links we compare CRE_gene name so should be ok to leave as is

## 4a. Reformat significant CP links

### Functions

In [66]:
### Classify links
classify_links <- function(celltype, cic_fp, cic_outdir){
    # Write file of all unique CREs in cicero links
    df <- read.table(cic_fp, sep='\t')
    sites1 <- df[,c(1,2,3)]
    sites2 <- df[,c(4,5,6)]
    colnames(sites2) <- c("V1","V2","V3")
    chr_names <- c(paste("chr",seq(1:22),sep=''),'chrX','chrY')
    all_sites <- rbind(sites1[sites1$V1 %in% chr_names,],sites2[sites2$V1 %in% chr_names,])
    all_sites_list <- paste(all_sites$V1,all_sites$V2,all_sites$V3,sep='_')
    all_sites_list <- sort(unique(all_sites_list))

    fin_df <- as.data.frame(str_split_fixed(all_sites_list,'_',n=3))
    print(paste('Number of unique peaks: ', dim(fin_df)[1]))
    out_fp <- file.path(cic_outdir,sprintf('%s_links.unique_peaks.bedpe',celltype))
    write.table(fin_df, out_fp, sep='\t', row.names=FALSE, col.names=FALSE, quote=FALSE)
    
    # Overlap peaks with gene promoters
    cic_peaks <- out_fp
    overlap_fp <- file.path(cic_outdir, sprintf('%s_links.unique_peaks.gencodev32_TSS500.overlap.bed',celltype))
    cmd <- paste('bedtools intersect -a', cic_peaks, '-b', TSS_ref_fp, '-wa -wb >', overlap_fp, sep=' ')
    system(cmd)
    
    # Read in gene overlaps and classify all Cicero links
    gene_overlaps <- read.table(overlap_fp, sep='\t')
    gene_overlaps$peak1 <- paste(gene_overlaps$V1, gene_overlaps$V2, gene_overlaps$V3, sep='-')
    print(paste('Number of Cicero peaks that overlap gene TSS regions: ',dim(gene_overlaps)[1]))
    print(paste('Number of unique Cicero peaks that overlap at least one promoter', length(unique(gene_overlaps$peak1))))
    
    # Make a reference for each peak (which genes it overlaps)
    get_genes <- function(peak, gene_overlaps){
        gene <- gene_overlaps[gene_overlaps$peak1 == peak,7]
        return(paste(gene, collapse=","))
    }
    unique_peaks <- unique(gene_overlaps$peak1)
    genes_key <- unlist(lapply(unique_peaks, get_genes, gene_overlaps))
    names(genes_key) <- unique_peaks

    # Add in promoter classifications to all the links
    get_overlap_prom <- function(peak, genes_key){
        if (peak %in% names(genes_key)){
            return(genes_key[peak])
        } else {
            return(NA)
        }
    }
    df$peak1 = paste(df$V1, df$V2, df$V3, sep='-')
    df$peak2 = paste(df$V4, df$V5, df$V6, sep='-')
    df$prom1 = unlist(lapply(df$peak1, get_overlap_prom, genes_key))
    df$prom2 = unlist(lapply(df$peak2, get_overlap_prom, genes_key))

    # Classify each link based on prom1 and prom2
    classify_link <- function(df_row){
        prom1 = df_row[['prom1']]
        prom2 = df_row[['prom2']]
        if (is.na(prom1) && is.na(prom2)){
            return('CC')
        } else if (!is.na(prom1) && is.na(prom2)){
            return('CP')
        } else if (!is.na(prom1) && !is.na(prom2)){
            return('PP')
        } else if (is.na(prom1) && !is.na(prom2)){
            return('CP')
        }
    }
    df$class = apply(df,1, classify_link)
    print(table(df$class))
    out_fp = file.path(cic_outdir,sprintf('%s_links.wClass.bedpe',celltype))
    write.table(df[,-8],out_fp, sep='\t', col.names=FALSE, row.names=FALSE, quote=FALSE)
    print('')
}

In [67]:
### Cut down links to CP (CRE-gene promoter links)
cut_links_to_CP <- function(celltype, cic_outdir){
    # Read in classified cicero outputs
    cic_fp <- file.path(cic_outdir,sprintf('%s_links.wClass.bedpe',celltype))
    print(cic_fp)
    cic_df <- read.table(cic_fp, sep='\t')
    colnames(cic_df) <- c('chrom1', 'start1', 'end1', 'chrom2', 'start2', 'end2', 'Cicero_score', 
                          'peak1', 'peak2', 'prom1', 'prom2', 'class')

    # Cut down to just CP links
    cic_cp <- cic_df[cic_df$class == 'CP',]
    print(paste('CP links: ', dim(cic_cp)[1], '/', dim(cic_df)[1], sep=''))
    return(cic_cp)
}

In [85]:
### Reformat links to be: CRE coords, gene coords, gene name, score
reformat_links <- function(cic_cp, celltype, cic_outdir, max_dist=250000){
    # Extract gene name from either prom1 or prom2 column
    get_gene <- function(row){
        gene1 <- row[["prom1"]]
        gene2 <- row[["prom2"]]
        if (is.na(gene1)){
            return(gene2)
        } else {
            return(gene1)
        }
    }
    cic_cp$gene <- apply(cic_cp, 1, get_gene)

    # Separate out multigene rows into new df for further work
    cic_cp1 <- cic_cp[!grepl(',', cic_cp$gene),]
    cic_cp2 <- cic_cp[grepl(',', cic_cp$gene),]
    multigenes <- cic_cp2$gene
    cic_cp2_split <- data.frame()

    # Go through every row from multigene rows and split up each gene to a new row
    for(i in seq(1,dim(cic_cp2)[1])){
        genes <- unlist(strsplit(multigenes[[i]], ','))
        df_row <- cic_cp2[i,seq(1,12)]
        new_rows <- rbind(df_row, rep(df_row[rep(1,length(genes)-1),]))
        new_rows$gene <- genes
        cic_cp2_split <- rbind(cic_cp2_split, new_rows)
    }

    # Combine split multigenes with the one gene rows in cic_cp1
    cic_cp_combo <- rbind(cic_cp1, cic_cp2_split)

    # Swap coords for links where prom2 has the gene
    cic_left <- cic_cp_combo[!is.na(cic_cp_combo$prom1),]
    cic_right <- cic_cp_combo[!is.na(cic_cp_combo$prom2),]
    cic_left <- cic_left[,c(4,5,6,1,2,3,seq(7,13))]
    colnames(cic_left) <- c('chrom1', 'start1', 'end1', 'chrom2', 'start2', 'end2', 'Cicero_score', 
                          'peak1', 'peak2', 'prom1', 'prom2', 'class', 'gene')
    cic_fin <- rbind(cic_right, cic_left)

    # Make final df for output (sort by CRE positions, cut out unnecessary cols, replace
    # P peak coords with gene TSS, to match all other links final outputs)
    cic_fin <- cic_fin[order(cic_fin$chrom1,cic_fin$start1),]
    cic_fin2 <- cic_fin[,c(seq(1,6),13,7)] #order: coords, gene, score
    print(paste('Number of distinct CP CRE-gene links: ',dim(cic_fin2)[1], sep=''))

    # Add final filters (TPM > 1, distance < 250kb)
    tpm_fp <- paste0(tpm_prefix,celltype,tpm_suffix)
    tpm1_genes <- scan(tpm_fp, what='', sep='\t')
    cic_fin2$distance <- apply(cic_fin2, 1, calc_link_distance)
    cic_fin2 <- cic_fin2 %>% mutate(expressed=ifelse(gene %in% tpm1_genes,'y','n')) %>%
                        subset(expressed == 'y' & distance < max_dist) %>%
                        select(-c(distance,expressed))
    print(paste('Number of distinct CP CRE-gene links passing TPM and dist filters: ',dim(cic_fin2)[1], sep=''))

    # Output to a file!
    out_fp <- file.path(cic_outdir, sprintf('%s_links.CP_reformat.bedpe',celltype))
    write.table(cic_fin2, out_fp, sep='\t', row.names=FALSE, col.names=FALSE, quote=FALSE)
}

### Run functions on all celltypes to reformat links

In [69]:
# Run function to create classified output file (input is already FDR thresholded)
tic()
for (celltype in celltypes){
    cic_fp <- paste0(cic_prefix, celltype, cic_suffix)
    print(cic_fp)
    classify_links(celltype, cic_fp, cic_outdir)
}
toc()

[1] "/nfs/lab/projects/multiomic_islet/outputs/multiome/cRE-gene_links/cicero/241118_union_peaks/Cicero_links.beta.above0.05.dedup.bedpe"
[1] "Number of unique peaks:  55258"
[1] "Number of Cicero peaks that overlap gene TSS regions:  7902"
[1] "Number of unique Cicero peaks that overlap at least one promoter 7070"

   CC    CP    PP 
65274 31376 10047 
[1] ""
[1] "/nfs/lab/projects/multiomic_islet/outputs/multiome/cRE-gene_links/cicero/241118_union_peaks/Cicero_links.alpha.above0.05.dedup.bedpe"
[1] "Number of unique peaks:  55460"
[1] "Number of Cicero peaks that overlap gene TSS regions:  8472"
[1] "Number of unique Cicero peaks that overlap at least one promoter 7565"

   CC    CP    PP 
59980 36797 11437 
[1] ""
[1] "/nfs/lab/projects/multiomic_islet/outputs/multiome/cRE-gene_links/cicero/241118_union_peaks/Cicero_links.delta.above0.05.dedup.bedpe"
[1] "Number of unique peaks:  65845"
[1] "Number of Cicero peaks that overlap gene TSS regions:  9962"
[1] "Number of unique Cicero pe

In [70]:
# Cut down Cicero outputs to CP links and reformat
tic()
for (celltype in celltypes){
    print(celltype)
    cic_cp <- cut_links_to_CP(celltype, cic_outdir)
    reformat_links(cic_cp, celltype, cic_outdir)
    print('')
}
toc()

[1] "beta"
[1] "/nfs/lab/projects/multiomic_islet/outputs/revisions_outputs/cRE-gene_links/links/Cicero/beta_links.wClass.bedpe"
[1] "CP links: 31376/106697"
[1] "Number of distinct CP CRE-gene links: 35362"
[1] "Number of distinct CP CRE-gene links passing TPM and dist filters: 14193"
[1] ""
[1] "alpha"
[1] "/nfs/lab/projects/multiomic_islet/outputs/revisions_outputs/cRE-gene_links/links/Cicero/alpha_links.wClass.bedpe"
[1] "CP links: 36797/108214"
[1] "Number of distinct CP CRE-gene links: 41531"
[1] "Number of distinct CP CRE-gene links passing TPM and dist filters: 16618"
[1] ""
[1] "delta"
[1] "/nfs/lab/projects/multiomic_islet/outputs/revisions_outputs/cRE-gene_links/links/Cicero/delta_links.wClass.bedpe"
[1] "CP links: 26466/85113"
[1] "Number of distinct CP CRE-gene links: 29619"
[1] "Number of distinct CP CRE-gene links passing TPM and dist filters: 12511"
[1] ""
[1] "gamma"
[1] "/nfs/lab/projects/multiomic_islet/outputs/revisions_outputs/cRE-gene_links/links/Cicero/gamma_link

## 4b. Process non-significant links to CP bedpe files with gene names
Steps:
1. Deduplicate links (remove repeated links, generally where CREs are swapped)
2. Overlap all CREs with a gene TSS file and then classify links as CC, CP, PP 
3. Reformat CP links to have gene names listed and separate lines, also with gene coords (just using TSS500 coords here)
4. Apply link filters (TPM > 1, dist < 250kb)

### Deduplicate all links first

In [72]:
### Function to remove duplicate connections, from all links file -- modified for parallel use and logging
run_dedup_all_links <- function(celltype, outdir, log_fp){
    # Read in all_links file
    write(paste(celltype, 'start', Sys.time()), file=log_fp, append=TRUE)
    all_links_fp <- paste0(cic_prefix, celltype, cic_suffix2)
    out_df_cut <- read.table(all_links_fp, sep='\t')
    
    # Remove duplicated links (same peaks and score, diff order)
    get_ordered_peaks <- function(row){
        if (row[2] < row[5]){
            peak1 = paste(row[[1]], as.character(row[[2]]), as.character(row[[3]]), sep='-')
            peak2 = paste(row[[4]], as.character(row[[5]]), as.character(row[[6]]), sep='-')
        } else {
            peak1 = paste(row[[4]], as.character(row[[5]]), as.character(row[[6]]), sep='-')
            peak2 = paste(row[[1]], as.character(row[[2]]), as.character(row[[3]]), sep='-')
        }
        return(paste(peak1,peak2,sep='_'))
    }

    out_df_cut$ordered_peaks = apply(out_df_cut,1,get_ordered_peaks)
    out_df_fin = out_df_cut[!duplicated(out_df_cut$ordered_peaks),]

    # Output the thresholded and dedup df
    out_fp2 = file.path(outdir,sprintf('Cicero_links.%s.all.dedup.bedpe',celltype))
    write.table(out_df_fin, out_fp2, sep='\t', col.names=FALSE, row.names=FALSE, quote=FALSE)
    write(paste(celltype, 'done', Sys.time()), file=log_fp, append=TRUE)
}

In [73]:
cic_outdir2 <- file.path(cic_outdir, 'nonsig_links')
cic_suffix2 <- '.all.bedpe'

In [74]:
log_fp <- '/nfs/lab/projects/multiomic_islet/outputs/revisions_outputs/cRE-gene_links/links/Cicero/nonsig_links/log.txt'

# Deduplicate all links -- took about 20 min for beta cells
write(paste('Deduplicating links:', Sys.time()), file=log_fp, append=TRUE)
mclapply(celltypes, run_dedup_all_links, cic_outdir2, log_fp, mc.cores=5)
write("\n", file=log_fp, append=TRUE)

[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

[[4]]
NULL

[[5]]
NULL

[[6]]
NULL

### Classify links

In [77]:
### Classify links -- modified for parallel use and logging
run_classify_links <- function(celltype, cic_outdir, log_fp){
    write(paste(celltype, 'start', Sys.time()), file=log_fp, append=TRUE)
    cic_fp <- file.path(cic_outdir,sprintf('Cicero_links.%s.all.dedup.bedpe',celltype))
    classify_links(celltype, cic_fp, cic_outdir)
    write(paste(celltype, 'done', Sys.time()), file=log_fp, append=TRUE)
}

In [78]:
# Create classified output file (same functions, diff outdir) -- took 2.5 hours to run on beta cells
write(paste('Classifying links:',Sys.time()), file=log_fp, append=TRUE)
mclapply(celltypes, run_classify_links, cic_outdir2, log_fp, mc.cores=5)
write("\n", file=log_fp, append=TRUE)

[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

[[4]]
NULL

[[5]]
NULL

[[6]]
NULL

### Select for CP links, add gene name and do final formatting

In [86]:
### Function to run the cut links to CP and reformat functions -- for parallel use
run_reformat_CP <- function(celltype, outdir, log_fp){
    write(paste(celltype, 'start', Sys.time()), file=log_fp, append=TRUE)
    cic_cp <- cut_links_to_CP(celltype, outdir)
    reformat_links(cic_cp, celltype, outdir)
    write(paste(celltype, 'done', Sys.time()), file=log_fp, append=TRUE)
}

In [ ]:
# Cut down Cicero outputs to CP links and reformat (same functions, diff outdir -- took 32 hours to run on beta cells
write(paste('Reformatting CP links:',Sys.time()), file=log_fp, append=TRUE)
mclapply(celltypes, run_reformat_CP, cic_outdir2, log_fp, mc.cores=5)
write(paste('Done',Sys.time()), file=log_fp, append=TRUE)

### Remove sig (score > 0.02) links from files

In [ ]:
### Function to filter Cicero background links reference using R, not bash
### Subset for links with score <= 0.02, select for 1Mb distance max
### These files should already be in a bedpe format!
filter_cicero_background <- function(celltype, cic_outdir, log_fp, score_threshold=0.02){
    write(paste(celltype, 'start', Sys.time()), file=log_fp, append=TRUE)

    # Read in the processed background bedpe file
    cic_fp <- file.path(cic_outdir,sprintf('%s_links.CP_reformat.bedpe',celltype))
    all_links <- read.table(cic_fp, sep='\t', header=TRUE)
    colnames(all_links) <- c('CRE_chr','CRE_start','CRE_end','gene_chr','gene_start','gene_end','gene','score')

    # Remove links passing the significance threshold (0.02) and which are not NaN
    all_links_cut <- all_links[all_links$score <= score_threshold,]    
    filt_fp <- str_replace(cic_fp, "CP_reformat", "final_filt")
    write.table(all_links_cut, filt_fp, sep='\t', row.names=FALSE, col.names=FALSE, quote=FALSE)
    write(paste(celltype, 'done', Sys.time()), file=log_fp, append=TRUE)
}

In [ ]:
# Finally remove sig links (score > 0.02)
write(paste('Removing sig links:',Sys.time()), file=log_fp, append=TRUE)
mclapply(celltypes, filter_cicero_background, cic_outdir2, log_fp, mc.cores=5)
write(paste('Done',Sys.time()), file=log_fp, append=TRUE)